In [0]:
#bronze_db.Stores_bronze
#bronze_db.Sales_bronze
#bronze_db.Products_bronze

In [0]:
#remove invalid rows: missing product_id, store_id, or quantity
Stores_bronze = spark.read.table("bronze_db.Stores_bronze")
display(Stores_bronze)

In [0]:
#Remove invalid rows: missing product_id, store_id, or quantity
Remove_invalid_Stores = Stores_bronze.na.drop()
display(Remove_invalid_Stores)


In [0]:
Remove_invalid_Stores.write.format("delta").mode("Overwrite").saveAsTable("silver_db.cleand_stores_data")

In [0]:
Products_bronze = spark.read.table("bronze_db.Products_bronze")
display(Products_bronze)


In [0]:
Remove_invalid_Products = Products_bronze.na.drop()
display(Remove_invalid_Products)


In [0]:
Remove_invalid_Products.write.format("delta").mode("Overwrite").saveAsTable("silver_db.cleand_products_data")

In [0]:
sales_bronze_remove = spark.read.table("bronze_db.Sales_bronze").na.drop()
display(sales_bronze_remove)

In [0]:
#Remove duplicate sale_id records
Remove_duplicates_sales = sales_bronze_remove.dropDuplicates(["sale_id"])
display(Remove_duplicates_sales)


In [0]:
#Convert data types: quantity to integer, price to integer, sale_timestamp to timestamp

from pyspark.sql.functions import col
Convert_Data_type = Remove_duplicates_sales.withColumn("Quantity_int", col("quantity").cast("int"))\
    .withColumn("price_int", (col("price")).cast("int"))\
    .withColumn("Sale_timestamp_to_timestamp", (col("sale_timestamp")).cast("timestamp"))



In [0]:

#eate calculated columns: total_amount = quantity × price, sale_date from sale_timestam
from pyspark.sql import functions as F
total_amount = Convert_Data_type.withColumn("total_amount", F.col("Quantity_int") * F.col("price_int"))\
    .withColumn("sale_date", F.to_date(F.col("sale_timestamp")))
total_amount.display()

In [0]:
#validate cleaned data: check for negative quantity, zero price, future timestamp

from pyspark.sql import functions as F

validated_data_df_sales = total_amount.filter(
    (F.col("Quantity_int") >= 0) &
    (F.col("price_int") > 0) 
)
display(validated_data_df_sales)


In [0]:
validated_data_df_sales.write.format("delta").mode("Overwrite").saveAsTable("silver_db.cleand_sales_data")